<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M02/M02_Lab2_System_Messages_Templates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🎭 M02 Lab 2 — System Messages, Templates & Model Comparison</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Beginner &nbsp;|&nbsp; ⏱ Time: ~12 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Craft effective <b>system messages</b> and role prompts</li>
    <li>Build reusable <b>prompt templates</b> with parameterization</li>
    <li>Compare <b>GPT vs Gemini</b> outputs side by side</li>
    <li>Understand how system messages shape model behavior</li>
  </ol>
</div>

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai tiktoken google-genai
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from openai import OpenAI
from google import genai
from dads5250 import setup_openai, setup_gemini, show_response, show_info, compare_responses, show_expected, quiz

# This reads your API keys from Colab Secrets automatically
client = setup_openai()
gemini = setup_gemini()

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ System Messages & Role Prompts</h2>
</div>

The **system message** is the most powerful tool for controlling model behavior. It sets the model's persona, tone, constraints, and rules *before* the user interacts.

Think of it as the "backstage instructions" to an actor — the audience (user) never sees them, but they shape every response.

In [ ]:
# ============================================================
# 🎭 Three Personas, Same Question
# ============================================================

personas = {
    "🎓 Formal Professor": "You are a formal university professor. Use academic language and cite concepts precisely.",
    "😊 Friendly Tutor": "You are a friendly tutor who explains things simply, using analogies and everyday language.",
    "⚡ Concise Expert": "You are a concise AI expert. Answer in 2-3 bullet points maximum. No fluff.",
}

question = "What is prompt engineering?"
results = {}

for name, system_msg in personas.items():
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": question},
        ],
        max_tokens=200,
    )
    results[name] = r.choices[0].message.content.strip()

compare_responses(results)

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Prompt Templates</h2>
</div>

Reusable prompt templates let you **parameterize** prompts for different inputs. Instead of writing a new prompt every time, you create a template with placeholders.

This is a key pattern in production AI — think of it as the "function" equivalent for prompts:

In [ ]:
# ============================================================
# 🔧 Reusable Classification Template
# ============================================================

def classify_with_template(text: str, categories: list[str]) -> str:
    """Reusable classification prompt template."""
    cats = ", ".join(categories)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": f"Classify the following text into one of these categories: {cats}. Respond with just the category name."},
            {"role": "user", "content": text},
        ],
        max_tokens=20,
    )
    return response.choices[0].message.content.strip()

# Test with customer support tickets
samples = [
    "My laptop screen cracked after dropping it",
    "How do I reset my password?",
    "I want a refund for my order #12345",
    "Do you ship internationally?",
]
categories = ["Hardware Issue", "Account Help", "Billing/Refund", "General Inquiry"]

print("📋 Customer Support Ticket Classification\n")
for s in samples:
    result = classify_with_template(s, categories)
    print(f"   📧 \"{s}\"")
    print(f"      → {result}\n")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ GPT vs Gemini: Side-by-Side</h2>
</div>

Different LLM providers have different strengths. Let's compare how OpenAI's **GPT** and Google's **Gemini** handle the same prompts.

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Note:</b> This requires a <code>GEMINI_API_KEY</code> in your Colab Secrets. If you don't have one, you can get a free key at <a href="https://aistudio.google.com/apikey">Google AI Studio</a>.
</div>

# ============================================================
# 🔄 GPT vs Gemini: Single Prompt Comparison
# ============================================================

prompt = "Explain the concept of 'attention' in transformer models in 3 sentences."

gpt_r = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
)

gemini_r = gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)

compare_responses({
    "🟢 GPT-4.1-mini": gpt_r.choices[0].message.content,
    "🔵 Gemini 2.5 Flash": gemini_r.text,
})

In [ ]:
<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Multi-Prompt Model Battle</h2>
</div>

Let's run 5 different prompts on both models and see how they compare across different task types:

# ============================================================
# ⚔️ 5-Prompt Model Battle: GPT vs Gemini
# ============================================================

test_prompts = [
    ("🌍 Translation",    "Translate 'Good morning, how are you?' to French, Spanish, and Japanese."),
    ("✍️ Creative",       "Write a haiku about machine learning."),
    ("📊 Factual",        "What are the top 3 causes of climate change? Be concise."),
    ("🧮 Reasoning",      "If a train leaves at 3pm going 60mph and another at 4pm going 80mph, when does the second catch up? Think step by step."),
    ("📝 Summarization",  "Summarize the concept of supply and demand in exactly 2 sentences."),
]

from IPython.display import display, HTML

for label, prompt in test_prompts:
    gpt_r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )
    gemini_r = gemini.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    display(HTML(f'<h4 style="color:#001a70; margin:16px 0 4px;">{label}</h4>'))
    compare_responses({
        "🟢 GPT-4.1-mini": gpt_r.choices[0].message.content,
        "🔵 Gemini 2.5 Flash": gemini_r.text,
    })

In [ ]:
**📝 Your Observations:** *(double-click to edit this cell)*

| Prompt Type | Better Model | Why? |
|------------|-------------|------|
| Translation | | |
| Creative | | |
| Factual | | |
| Reasoning | | |
| Summarization | | |

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What is the purpose of a system message?",
        "options": [
            "To authenticate the API key",
            "To set the model's persona, constraints, and behavior before the user interacts",
            "To count tokens in the prompt",
            "To select which model to use"
        ],
        "answer": 1,
        "explanation": "System messages define the assistant's role, tone, and constraints before the conversation begins. They're the primary tool for controlling model behavior."
    },
    {
        "q": "Why are prompt templates useful in production?",
        "options": [
            "They make the model faster",
            "They let you reuse the same prompt structure with different inputs",
            "They reduce API costs",
            "They're required by OpenAI"
        ],
        "answer": 1,
        "explanation": "Templates parameterize prompts so you can swap inputs without rewriting the whole prompt — essential for building scalable AI applications."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Build a Custom Chatbot Persona

Create a system message for a **restaurant recommendation chatbot**. It should follow specific rules you define.

In [ ]:
# ============================================================
# 🔧 Exercise: Restaurant Recommendation Chatbot
# ============================================================
# Replace each ----- with the correct value

# Step 1: Write a system message with at least 3 rules
restaurant_system = """You are a -----.
# 👆 YOUR CODE: Define the persona (e.g., "friendly Boston restaurant recommender")

RULES:
- -----
- -----
- -----
"""
# 👆 YOUR CODE: Add 3 rules (e.g., always suggest 2-3 options, include price range, etc.)

# Step 2: Test with customer questions
test_questions = [
    "I'm looking for a good Italian place for a date night.",
    "What's a cheap lunch spot near campus?",
    "I'm vegan — any suggestions?",
]

for q in test_questions:
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "-----", "content": restaurant_system},   # What role?
            {"role": "-----", "content": q},                    # What role?
        ],
        max_tokens=200,
    )
    print(f"🧑 Customer: {q}")
    print(f"🍽️ Bot: {r.choices[0].message.content.strip()}\n")

**📝 Your Observations:** *(double-click to edit this cell)*

1. Did the chatbot follow all 3 rules you wrote? _[Your answer]_

2. How did it handle the vegan constraint — did it combine your rules with the dietary restriction? _[Your answer]_

3. What would you add to the system message to improve it? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these experiments on your own:</p>
  <ol style="font-size: 14px;">
    <li>Create a prompt template that generates <b>product descriptions</b> — takes product name, features, and target audience as inputs</li>
    <li>Test the same system message on GPT and Gemini — do they follow rules equally well?</li>
    <li>Try "jailbreaking" your system message — ask the model to ignore its rules. How robust is it?</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You've learned how to control model behavior with system messages, build reusable templates, and compare models.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>System messages</b> control persona, tone, constraints, and output format</li>
      <li><b>Prompt templates</b> make prompts reusable and parameterized for production</li>
      <li><b>Different models</b> have different strengths — always test and compare</li>
      <li>Explicit <b>rules</b> in system messages create guardrails for safe AI behavior</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Assignment M02:</b> Design a prompt for a real-world task. Test 3 techniques (zero-shot, few-shot, CoT). Submit prompts + outputs + 1-paragraph analysis.</p>
</div>

In [ ]:
test_prompts = [
    "Translate 'Good morning, how are you?' to French, Spanish, and Japanese.",
    "Write a haiku about machine learning.",
    "What are the top 3 causes of climate change? Be concise.",
    "If a train leaves at 3pm going 60mph and another at 4pm going 80mph, when does the second catch up? Think step by step.",
    "Summarize the concept of supply and demand in exactly 2 sentences.",
]

for i, prompt in enumerate(test_prompts, 1):
    gpt_r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )
    gemini_r = gemini.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    print(f"--- Prompt {i} ---")
    compare_responses({
        "GPT-4.1-mini": gpt_r.choices[0].message.content,
        "Gemini 2.5 Flash": gemini_r.text,
    })
    print()

**Your Observations:** (double-click to edit)

| Prompt Type | Better Model | Why? |
|------------|-------------|------|
| Translation | | |
| Creative | | |
| Factual | | |
| Reasoning | | |
| Summarization | | |

---
## Summary

- **System messages** control persona, tone, and constraints
- **Prompt templates** make prompts reusable and parameterized
- **Model comparison**: GPT and Gemini have different strengths — always test both

---

## Assignment M02

Design a prompt for a **real-world task** (customer support, data extraction, email drafting, etc.).

**Test 3 techniques:** zero-shot, few-shot, chain-of-thought

**Submit:**
- Your 3 prompt versions + outputs
- Analysis (1 paragraph): Which technique worked best and why?